In [0]:
# Load the bronze order, order_items, and products tables
orders = spark.read.table("revenue_operations.bronze.orders")
order_items = spark.read.table("revenue_operations.bronze.order_items")
payments = spark.read.table("revenue_operations.bronze.payments")
customers = spark.read.table("revenue_operations.bronze.customers")
products = spark.read.table("revenue_operations.bronze.products")
sellers = spark.read.table("revenue_operations.bronze.sellers")

### Silver Orders

In [0]:
orders.show(5)
orders.printSchema()

In [0]:
# Checking for duplicate rows
duplicate_count = orders.count() - orders.dropDuplicates().count()
print(f"Duplicate rows in orders: {duplicate_count}")

print(f"Total rows in orders: {orders.count()}")

from pyspark.sql.functions import col as column, sum
for col in orders.columns:
    print(f"Distinct value count in {col}: {orders.select(col).distinct().count()}")

print("\n")
for col in orders.columns:
    print(f"Number of null values in {col}: {orders.filter(column(col).isNull()).count()}")


`order_id` **is a valid primary key as it has all unique values and is not missing any values.**

In [0]:
orders.select("order_status").distinct().show()

In [0]:
from pyspark.sql.functions import col as column

invalid_purchased = orders.filter(column("order_purchase_timestamp") >= column("order_approved_at"))
invalid_count = invalid_purchased.count()

if invalid_count == 0:
    print("All orders: purchase timestamp is earlier than to approved timestamp")
else:
    print(f"Warning! Found {invalid_count} orders where purchase timestamp is later than approved timestamp")

invalid_approved = orders.filter(column("order_approved_at") >= column("order_delivered_carrier_date"))
invalid_count1 = invalid_approved.count()

if invalid_count1 == 0:
    print("All orders: approved timestamp is earlier than to delivered carrier timestamp")
else:
    print(f"Warning! Found {invalid_count1} orders where approved timestamp is later than delivered carrier timestamp")

invalid_delivered = orders.filter(column("order_delivered_carrier_date") >= column("order_delivered_customer_date"))
invalid_count2 = invalid_delivered.count()

if invalid_count2 == 0:
    print("All orders: delivered carrier timestamp is earlier than to delivered customer timestamp")
else:
    print(f"Warning! Found {invalid_count2} orders where delivered carrier timestamp is later than delivered customer timestamp")

invalid_delivered_purchased = orders.filter(column("order_delivered_customer_date") <= column("order_purchase_timestamp"))
invalid_count3 = invalid_delivered_purchased.count()

if invalid_count3 == 0:
    print("All orders: delivered customer timestamp is later than to purchase timestamp")
else:
    print(f"Warning! Found {invalid_count3} orders where delivered customer timestamp is earlier than purchase timestamp")

invalid_delivered_estimated = orders.filter(column("order_estimated_delivery_date") <= column("order_purchase_timestamp"))
invalid_count4 = invalid_delivered_estimated.count()

if invalid_count4 == 0:
    print("All orders: estimated delivery date is later than purchase timestamp")
else:
    print(f"Warning! Found {invalid_count4} orders where estimated delivery date is earlier than or equal to purchase timestamp")

In [0]:
# Foreign key check
print(f"Total number of unique customers in customer dataset : {customers.select("customer_id").distinct().count()}")
print(f"Total number of customers in orders dataset: {orders.select("customer_id").distinct().count()}")

# Checking if every customers that exists in the orders dataset also exists in the customers dataset
missing_customers = orders.join(customers.select("customer_id"), on = "customer_id", how = "leftanti")
print(f"Number of customer in customers dataset that are not in orders dataset: {missing_customers.count()}")

Orders Silver Design Notes
- Source = revenue_operations.bronze.orders
- Target = revenue_operations.silver.orders
- Columns renamed = None
- Data type Changes = None
- Duplicate findings = None (No duplicate rows)
- Null findings = 160 null values in order_approved_at (needs to be flagged as pending)
                = 1783 null values in order_delivered_carrier_date (In transit, not yet delivered)
                = 2965 null values in order_delivered_customer_date (Some orders are in transit, and some orders have not been marked delivered by customers.)
- Primary Key - order_id
- Validation findings 

        - 1296 orders with purchase timestamp later than approved timestamp.
        - 1359 orders with approved timestamp later than delivered carrier timestamp.
        - 32 orders with delivered carrrier timestamp later than delivered customer timestamp.
        - All orders delivered customer timestamp is later than to purchase timestamp and estimated delivery date is later than purchase timestamp. 

- Foreign Key findings = All the customer_id in the orders dataset are in customer dataset(Parent). 
- Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status

In [0]:
print("Validation Summary: \n")
print(f"Bronze orders row count = {orders.count()}")
print(f"Bronze orders with no duplicated = {orders.dropDuplicates().count()}")
print(f"Total number of missing keys = {orders.select("order_id").filter(orders.order_id.isNull()).count()}")

print("\nInvalid Values:")
print(f"Number of invalid purchase time stamp: {invalid_count}")
print(f"Number of invalid order approved timestamp: {invalid_count1}")
print(f"Number of invalid delivered carrier timestamp: {invalid_count2}")

print(f"Number of cusstomer_id in orders dataset that does not exit in customers dataset (Parent table): {missing_customers.count()}")

### Silver Orders DataFrame

In [0]:
from pyspark.sql.functions import lit, when, sum
from pyspark.sql import functions as F

# copying bronze.orders into different dataframe to create silver.orders
silver_orders_df = orders.select('*')
silver_orders_df.show(5)

print("Before", silver_orders_df.count())
# Dropping Duplicates
silver_orders_df = silver_orders_df.dropDuplicates()
print("After", silver_orders_df.count())

# Adding audit columns
silver_orders_df = silver_orders_df.withColumn('source_file_name', lit("olist_orders_dataset.csv"))
silver_orders_df = silver_orders_df.withColumn('ingestion_timestamp', F.expr("current_timestamp() - INTERVAL 2 DAYS"))
silver_orders_df = silver_orders_df.withColumn('silver_processed_timestamp', F.current_timestamp())

silver_orders_df = silver_orders_df.withColumn('data_quality_status',
                                               when(column('order_id').isNull(), lit("missing_key")).
                                               when((column("order_purchase_timestamp") >= column("order_approved_at")), lit("invalid_purchase_timestamp")).
                                               when((column("order_approved_at") >= column("order_delivered_carrier_date")), lit("invalid_approved_timestamp")).
                                               when((column("order_delivered_carrier_date") >= column("order_delivered_customer_date")), lit("invalid_delivered_timestamp")).
                                               when((column("order_delivered_customer_date") <= column("order_purchase_timestamp")), lit("invalid_delivered_purchased_timestamp")).
                                               when((column("order_estimated_delivery_date") <= column("order_purchase_timestamp")), lit("invalid_delivered_estimated_timestamp")).
                                               otherwise(lit("valid")))

data_quality_status_count = silver_orders_df.groupBy('data_quality_status').count()
data_quality_status_count.orderBy(F.desc('count')).show()
silver_orders_df.show(5)


In [0]:
## Saving into silver.orders delta table
silver_orders_df.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.silver.orders")

### Silver Order Items

In [0]:
order_items.printSchema()

In [0]:
# Check for duplicate rows
duplicate_counts = order_items.count() - order_items.dropDuplicates().count()
print(f"Total number of duplicate rows = {duplicate_counts}")

In [0]:
# Check for uniqueness of order_id + order_item_id
print(f"Total rows in order_items dataset = {order_items.count()}")
print(f"Total unique row of order_id + order_item_id = {order_items.select('order_id', 'order_item_id').distinct().count()}")

In [0]:
from pyspark.sql.functions import col as column
# Check for null values
for col in order_items.columns:
    print(f"Number of null values in {col}: {order_items.filter(column(col).isNull()).count()}")

`order_id` + `order_item_id` **composite keys are valid because both has no missing vaules and are unique.**

In [0]:
order_items.select("order_item_id").distinct().show()

In [0]:
# Validation rules

# order_item_id >= 1
invalid_order_item_id = order_items.filter(column("order_item_id") < 1) 

invalid_count = invalid_order_item_id.count()
if invalid_count > 0:
    print(f"Number of order_item_id values less than 1: {invalid_count}")
else:
    print("All order_item_id values are greater than or equal to 1")

# price > 0
invalid_price = order_items.filter(column("price") <= 0)
invalid_price_count = invalid_price.count()
if invalid_price_count > 0:
    print(f"Number of price less than or equal to 0 : {invalid_price_count}")
else:
    print("All the prices are greater than 0.")

# freight_value >= 0
invalid_freight_value = order_items.filter(column("freight_value") < 0)
invalid_freight_value_count = invalid_freight_value.count()
if invalid_freight_value_count > 0:
    print(f"Number of freight_value less than 0: {invalid_freight_value_count}")
else:
    print("All freight_value are greater than or equal to 0.")

# Shipping_limit_date is valid timestamp
invalid_shipping_limit_date = order_items.filter(column("shipping_limit_date").isNull())
invalid_shipping_limit_date_count = invalid_shipping_limit_date.count()
if invalid_shipping_limit_date_count > 0:
    print(f"Number of null shipping_limit_date: {invalid_shipping_limit_date_count}")
else:
    print("All shipping_limit_date are valid timestamp")

# Check for unusual freight value to price
unusual_freight_value_to_price = order_items.filter(column("freight_value") > column("price"))
unusual_freight_value_to_price_count = unusual_freight_value_to_price.count()
print(f"Number of freight_value greater than price: {unusual_freight_value_to_price_count}")
print("\n Unusual freight_values to price are:")
unusual_freight_value_to_price.show()

In [0]:
# Foreign keys check

missing_order_id = order_items.join(orders.select("order_id"), on = "order_id", how = "leftanti")
print(f"Number of order_id in order_itmes dataset that are not in order dataset: {missing_order_id.count()}")

missing_product_id = order_items.select("product_id").join(products.select("product_id"), on = "product_id", how = "leftanti")
print(f"Number of product_id in order_items dataset that are not in products dataset: {missing_product_id.count()}")

missing_seller_id = order_items.select("seller_id").join(sellers.select("seller_id"), on = "seller_id", how = "leftanti")
print(f"Number of seller_id in order_id dataset that are not in sellers dataset: {missing_seller_id.count()}")



In [0]:
# Converting datatypes
from pyspark.sql.functions import col
order_items = order_items.withColumn("price", col("price").cast("decimal(18,2)")) \
                         .withColumn("freight_value", col("freight_value").cast("decimal(18,2)"))
order_items.printSchema()

Order Items Silver Design notes
- Source = revenue_operations.bronze.order_items
- Target = revenue_operations.silver.order_items
- Columns renamed = None
- Data type changes = price and freight_value changed from double to decimal
- Duplicate findings = No duplicate rows
- Null findings = None
- Composite Key = order_id + order_item_id
- Validation findings :

    - All order_item_id values are greater than or equal to 1
    - All the prices are greater than 0.
    - All freight_value are greater than or equal to 0.
    - All shipping_limit_date are valid timestamp
    - Number of freight_value greater than price: 4124
- Foreign-key findings: 

    - All the order_id in the order_items dataset are in the orders dataset (Parent table).
    - All the product_id in the order_items dataset are in the products dataset (Parent table).
    - All the seller_id in the order_items dataset are in the sellers dataset (Parent table).
- Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status


In [0]:
# Validation Summary
print("Validation Summary \n")
print(f"Bronze order_items row count : {order_items.count()}")
print(f"Bronze order_items with no duplicate: {order_items.dropDuplicates().count()}")
print(f"Missing keys count: {order_items.select("order_id", "order_item_id").filter(column("order_id").isNull() | column("order_item_id").isNull()).count()}")
print(f"Unusual Values: Freight_value greater than price count: {unusual_freight_value_to_price_count} ")
print("\nUnmatched Foreign Keys")
print(f"Missing order_id in orders dataset count: {missing_order_id.count()}")
print(f"Missing product_id in products dataset count: {missing_product_id.count()}")
print(f"Missing seller_id in sellers dataset count: {missing_seller_id.count()}")

### Silver Order Items Dataframe

In [0]:
# Copying order items into silver_order_items_df
silver_order_items_df = order_items.select("*")
silver_order_items_df.show(5)

print("Before: ", silver_order_items_df.count())
silver_order_items_df = silver_order_items_df.dropDuplicates()
print("After: ", silver_order_items_df.count())

silver_order_items_df.printSchema()

# Adding audit columns
silver_order_items_df = silver_order_items_df.withColumn('source_file_name', 
lit("olist_order_items_dataset.csv")).withColumn('ingestion_timestamp', F.expr("current_timestamp() - INTERVAL 2 DAYS")).withColumn('silver_processed_timestamp', F.expr("current_timestamp()"))

silver_order_items_df = silver_order_items_df.withColumn('data_quality_status', 
                                                         when(column('order_id').isNull() | column('order_item_id').isNull() | column('product_id').isNull() | column('seller_id').isNull(), lit('missing key')).
                                                         when((column("order_item_id") < 1) | (column("price") <= 0) | (column("freight_value") < 0), lit('invalid value')).
                                                         otherwise(lit('valid')))

silver_order_items_df_count = silver_order_items_df.groupBy('data_quality_status').count()
silver_order_items_df_count.orderBy('count', ascending = False).show()

silver_order_items_df.show(10)



In [0]:
# Saving as silver delta table
silver_order_items_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('revenue_operations.silver.order_items')
                                   

### Silver Payments

In [0]:
payments.printSchema()

In [0]:
# Duplicate rows count
duplicate_payments = payments.count() - payments.dropDuplicates().count()
print(f"Number of duplicate rows in payments dataset: {duplicate_payments}")

# Check uniqueness of order_id + payment_sequential
print(f"Total Number of rows in payments: {payments.count()}")
print(f"Total Number of unique order_id + payment_sequential in payments: {payments.select("order_id", "payment_sequential").distinct().count()}")

# Check for null values in columns
for col in payments.columns:
    print(f"Number of null values in {col}: {payments.filter(column(col).isNull()).count()}")

`order_id` + `payment_sequential` **are valid composite keys as they are not missing any values and are unique**

In [0]:
# Validation rules

# payment_squential >= 1
invalid_payment_sequential = payments.filter(column("payment_sequential") < 1).count()
print(f"Number of payment sequential less than 1: {invalid_payment_sequential}")

# payment_installments >= 0
invalid_payment_installments = payments.filter(column("payment_installments") < 0).count()
print(f"Number of payment installments that are less than 0: {invalid_payment_installments}")

# payment_value >= 0
invalid_payment_value = payments.filter(column("payment_value") < 0).count()
print(f"Number of payment values that are less than 0: {invalid_payment_value}")

payments.select("payment_type").distinct().show()

In [0]:
# Foreign Key check 

# Checking for every order_id in payments also exist in orders table
missing_order_id_orders = payments.select("order_id").join(orders.select("order_id"), on="order_id", how="leftanti")
print(f"Number of order_id in payments dataset that are not in orders dataset: {missing_order_id_orders.count()}")

In [0]:
# Changing data type
payments = payments.withColumn("payment_value", column("payment_value").cast("decimal(18,2)"))
payments.printSchema()

Payments Silver design notes
- Source = revenue_operations.bronze.payments
- Target = revenue_operations.silver.payments
- Columns renamed = None
- Data-type changed:
    - payment_value data type changed from double to decimal.
- Null findings = None
- Composite keys = order_id + payment_sequential
- Validation findings:
    - All payments sequential greater than 0.
    - All payment installments greater than or equal to 0.
    - All payment values greater than or equal to 0.
- Foreign Key findings = All order_id in payements dataset are in orders dataset (Parent table).
- Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status


In [0]:
# Validation Summary
print("Validation Summary \n")
print(f"Bronze payments row count : {payments.count()}")
print(f"Bronze payments with no duplicate: {payments.dropDuplicates().count()}")
print(f"Missing keys count: {payments.select("order_id", "payment_sequential").filter(column("order_id").isNull() | column("payment_sequential").isNull()).count()}")
print(f"Invalid Value: payment_sequential less than 1 count: {invalid_payment_sequential} ")
print(f"Invalid Value: payment_installments less than 0 count: {invalid_payment_installments} ")
print(f"Invalid Value: payment_value less than 0 count: {invalid_payment_value} ")
print("\nUnmatched Foreign Keys")
print(f"Missing order_id in orders dataset count: {missing_order_id_orders.count()}")


### Silver Payments Dataframe

In [0]:
# Copying the bronze table into dataframe
silver_payments_df = payments.select("*")

# Drop duplicates
print("Before : ", silver_payments_df.count())
silver_payments_df = silver_payments_df.dropDuplicates()
print("After : ", silver_payments_df.count())

silver_payments_df.printSchema()

# Adding audit columns
audit_columns = ['source_file_name', 'ingestion_timestamp', 'silver_processed_timestamp', 'data_quality_status']

silver_payments_df = silver_payments_df.withColumn("source_file_name", lit("olist_payments_dataset.csv")).withColumn("ingestion_timestamp", F.expr("current_timestamp() - INTERVAL 2 DAYS")).withColumn('silver_processed_timestamp', F.expr("current_timestamp()"))

silver_payments_df = silver_payments_df.withColumn("data_quality_status", 
                                                   when(column('order_id').isNull() | column('payment_sequential').isNull(), lit("missing key")).
                                                   when((column("payment_sequential") < 1) | (column("payment_installments") < 0) | (column("payment_value") < 0), lit("invalid_value")).
                                                   otherwise(lit("valid")))

silver_payments_audit_count = silver_payments_df.groupBy("data_quality_status").count()
silver_payments_audit_count.orderBy('count', ascending = False).show()

silver_payments_df.show(10)


In [0]:
# Writing into delta table
silver_payments_df.write.format('delta').mode("overwrite").saveAsTable("revenue_operations.silver.payments")

In [0]:
# Reading the silver table and checking
silver_orders = spark.read.table("revenue_operations.silver.orders")
silver_order_items = spark.read.table("revenue_operations.silver.order_items")
silver_payments = spark.read.table("revenue_operations.silver.payments")

In [0]:
print( "Number of rows in silver_orders",silver_orders.count())
silver_orders.printSchema()
silver_orders.groupBy("data_quality_status").count().orderBy('count', ascending = False).show()
silver_orders.show(10)


print("Number of rows in silver order items",silver_order_items.count())
silver_order_items.printSchema()
silver_order_items.groupBy("data_quality_status").count().orderBy('count', ascending = False).show()
silver_order_items.show(10)

print("Number of rows in silver payments", silver_payments.count())
silver_payments.printSchema()
silver_payments.groupBy("data_quality_status").count().orderBy('count', ascending = False).show()
silver_payments.show(10)